# Lesson 01: Introduction to MCP & Google AI Studio Setup

In this notebook, you will:
1. Install the official `mcp` Python SDK and `google-genai`.
2. Load your free `GEMINI_API_KEY` safely from **Google Colab Secrets** (or a local `.env` file).
3. Build a minimal `FastMCP` server in Python and inspect its registered tools.

In [ ]:
!pip install -q -U "mcp<2" google-genai python-dotenv

In [ ]:
import os
from google import genai

def get_gemini_client():
    api_key = None
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    if not api_key:
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except Exception:
            pass
        api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        print("GEMINI_API_KEY not found. Add it to Colab Secrets to enable live Gemini calls.")
        return None
    return genai.Client(api_key=api_key)

client = get_gemini_client()
print("Gemini client ready:", client is not None)

In [ ]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("StarterMCPServer")

@mcp.tool()
def integration_savings(apps: int, tools: int) -> dict:
    """Calculate M x N vs M + N integration savings."""
    custom = apps * tools
    standardized = apps + tools
    return {
        "custom_integrations": custom,
        "mcp_integrations": standardized,
        "reduction_pct": round((1 - standardized / custom) * 100, 1),
    }

print("Registered tool result:", integration_savings(5, 12))

In [ ]:
prompt = "In 2 sentences, explain why MCP is called the USB-C for AI applications."
if client:
    resp = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
    print(resp.text)
else:
    print("[Offline Demo] MCP provides a single open protocol connecting M AI apps to N tools, eliminating M x N custom glue code.")

### Challenge Exercise
Add a second `@mcp.tool()` named `summarize_topology(servers: list[str])` to `StarterMCPServer` and test calling it directly.